# Bullinger HTR: Clean Colab Entry
Read TRAINING.md before selecting a task. CPU: context. GPU: training/adaptation/reranking/test. Historical Drive data and checkpoints are required. Run one task at a time.

In [ ]:
TASK = None  # train10k / train20k / adapt / rerank / context / test
SEED = 44  # adaptation only

In [ ]:
from pathlib import Path
SOURCES={'train_10k_sampling.py': 'from pathlib import Path\nimport json,zipfile,subprocess,sys,time,os,shutil\nroot=Path(\'/content/drive/MyDrive/BullingerAblation\');work=Path(\'/content/bullinger-ablation\');work.mkdir(exist_ok=True)\nsubprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'transformers==4.51.0\',\'Pillow==12.2.0\',\'numpy==2.2.6\'])\nplan=work/(\'sampling-control-plan-\'+time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime()));plan.mkdir()\nwith zipfile.ZipFile(root/\'input/experiment-code.zip\') as z:\n for n in z.namelist():\n  if n.startswith(\'plan/\') and n.endswith(\'.json\'):(plan/Path(n).name).write_bytes(z.read(n))\n (work/\'prepare.py\').write_bytes(z.read(\'prepare.py\'))\nscript=work/\'run_sampling_control.py\'\nscript.write_text("\\"\\"\\"Train one manifest, or evaluate a saved checkpoint on its fixed test set.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport random\\nimport shutil\\nimport sys\\nimport time\\nfrom prepare import audit, dump\\n\\n\\ndef distance(a, b):\\n    previous = list(range(len(b)+1))\\n    for i, x in enumerate(a, 1):\\n        current = [i]\\n        for j, y in enumerate(b, 1):\\n            current.append(min(current[-1]+1, previous[j]+1, previous[j-1]+(x != y)))\\n        previous = current\\n    return previous[-1]\\n\\n\\ndef metrics(rows):\\n    out = {}\\n    for group in [\'overall\', \'de\', \'la\']:\\n        subset = [r for r in rows if group == \'overall\' or r[\'language\'] == group]\\n        chars = sum(len(r[\'reference\']) for r in subset)\\n        words = sum(len(r[\'reference\'].split()) for r in subset)\\n        ce = sum(distance(r[\'reference\'], r[\'prediction\']) for r in subset)\\n        we = sum(distance(r[\'reference\'].split(), r[\'prediction\'].split()) for r in subset)\\n        out[group] = dict(n=len(subset), character_errors=ce, reference_chars=chars, word_errors=we, reference_words=words, cer=ce/chars if chars else None, wer=we/words if words else None)\\n    return out\\n\\n\\ndef aligned_training_loss(logits, labels):\\n    \\"\\"\\"Decoder inputs are already shifted by VisionEncoderDecoderModel.\\"\\"\\"\\n    import torch.nn.functional as F\\n    return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),\\n                           labels.to(logits.device).reshape(-1), ignore_index=-100)\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\'--manifest\', type=Path, required=True)\\n    p.add_argument(\'--out\', type=Path, required=True)\\n    p.add_argument(\'--model\', default=\'microsoft/trocr-base-handwritten\')\\n    p.add_argument(\'--revision\', default=\'main\', help=\'Use immutable Hugging Face commit for cross-machine reproduction\')\\n    p.add_argument(\'--evaluate-only\', action=\'store_true\')\\n    p.add_argument(\'--device\', choices=[\'cuda\', \'mps\', \'cpu\'], default=\'cuda\')\\n    p.add_argument(\'--epochs\', type=int, default=5)\\n    p.add_argument(\'--batch-size\', type=int, default=2)\\n    p.add_argument(\'--resume-from\', type=Path)\\n    p.add_argument(\'--lr\', type=float, default=1e-5)\\n    p.add_argument(\'--max-length\', type=int, default=128)\\n    args = p.parse_args()\\n    if args.epochs < 1 or args.batch_size < 1 or args.max_length < 2 or args.lr <= 0:\\n        raise ValueError(\'Invalid training parameters\')\\n    if args.out.exists():\\n        raise ValueError(\'Output exists; choose a fresh run directory\')\\n    m = json.loads(args.manifest.read_text())\\n    ep = args.manifest.parent / m[\'evaluation_file\']\\n    evaluation = json.loads(ep.read_text())\\n    fingerprint = hashlib.sha256(json.dumps(evaluation, sort_keys=True).encode()).hexdigest()\\n    if fingerprint != m[\'evaluation_sha256\']:\\n        raise ValueError(\'Evaluation manifest changed\')\\n    split = dict(train=m[\'train\'], **evaluation)\\n    if any(v for d in audit(split).values() for v in d.values()):\\n        raise ValueError(\'Letter or line leakage\')\\n    for rows in split.values():\\n        for r in rows:\\n            if r[\'language\'] not in [\'de\', \'la\'] or not Path(r[\'image\']).is_file() or not Path(r[\'text\']).is_file():\\n                raise ValueError(f\'Invalid sample {r}\')\\n    os.environ[\'CUBLAS_WORKSPACE_CONFIG\'] = \':4096:8\'\\n    import numpy as np\\n    import torch\\n    from PIL import Image\\n    from transformers import TrOCRProcessor, VisionEncoderDecoderModel\\n    seed = m[\'seed\']\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.use_deterministic_algorithms(True)\\n    device = torch.device(args.device)\\n    if args.device == \'cuda\' and not torch.cuda.is_available():\\n        raise RuntimeError(\'CUDA unavailable; use an explicit --device cpu/mps if desired\')\\n    completed_checkpoints = sorted(args.resume_from.glob(\'checkpoint_epoch*/complete.json\'),key=lambda p:int(p.parent.name.replace(\'checkpoint_epoch\',\'\'))) if args.resume_from else []\\n    if args.resume_from and not completed_checkpoints: raise ValueError(\'No complete epoch checkpoints\')\\n    checkpoint = completed_checkpoints[-1].parent if completed_checkpoints else None\\n    source_model = checkpoint if args.resume_from else args.model\\n    load_revision = None if args.resume_from else args.revision\\n    processor = TrOCRProcessor.from_pretrained(source_model, revision=load_revision, use_fast=False)\\n    model = VisionEncoderDecoderModel.from_pretrained(source_model, revision=load_revision).to(device)\\n    tok = processor.tokenizer\\n    for config in [model.config, model.generation_config]:\\n        config.decoder_start_token_id = tok.cls_token_id\\n        config.pad_token_id = tok.pad_token_id\\n        config.eos_token_id = tok.sep_token_id\\n    labels = {}\\n    data_hash = hashlib.sha256()\\n    for rows in split.values():\\n        for r in rows:\\n            text = Path(r[\'text\']).read_text(encoding=\'utf-8\').strip()\\n            if not text:\\n                raise ValueError(\'Empty reference\')\\n            ids = tok(text).input_ids\\n            if len(ids) > args.max_length:\\n                raise ValueError(f\'Label would truncate ({len(ids)} tokens): {r[\\"text\\"]}; increase --max-length for ALL runs\')\\n            labels[r[\'image\']] = ids\\n            data_hash.update(Path(r[\'image\']).read_bytes())\\n            data_hash.update(text.encode())\\n    resume_state = None\\n    prior_history = []\\n    if args.resume_from:\\n        prior_cfg = json.loads((args.resume_from/\'config.json\').read_text())\\n        if prior_cfg.get(\'training_loss\') != \'explicit_aligned_seq2seq_ce_v1\':\\n            raise ValueError(\'Cannot resume misaligned training\')\\n        for key, value in dict(model=args.model, revision=args.revision, max_length=args.max_length, epochs=args.epochs, batch_size=args.batch_size, lr=args.lr, evaluation_sha256=fingerprint, dataset_sha256=data_hash.hexdigest(), manifest=m).items():\\n            if prior_cfg[key] != value:\\n                raise ValueError(f\'Resume protocol mismatch: {key}\')\\n        if prior_cfg[\'versions\'] != {x:importlib.metadata.version(x) for x in [\'torch\',\'transformers\',\'Pillow\',\'numpy\']}:\\n            raise ValueError(\'Resume library versions differ\')\\n        resume_state = torch.load(checkpoint/\'training_state.pt\', map_location=\'cpu\', weights_only=False)\\n        prior_history = [r for r in json.loads((args.resume_from/\'history.json\').read_text()) if r[\'epoch\']<=resume_state[\'epoch\']]\\n        completed = resume_state[\'epoch\']\\n        if [r[\'epoch\'] for r in prior_history] != list(range(1,completed+1)):\\n            raise ValueError(\'Checkpoint and history disagree; do not silently resume\')\\n    args.out.mkdir(parents=True)\\n    if args.resume_from:\\n        # Completed parent checkpoints stay read-only.\\n        dump(args.out/\'history.json\',prior_history)\\n        for r in prior_history:\\n            epoch=r[\'epoch\']\\n            for filename in [f\'draws_epoch{epoch}.json\',f\'validation_epoch{epoch}.jsonl\']:\\n                src=args.resume_from/filename\\n                if src.exists(): shutil.copy2(src,args.out/filename)\\n        dump(args.out/\'resume_provenance.json\',dict(parent=str(args.resume_from),completed_epoch=resume_state[\'epoch\'],replay=\'Restart incomplete next epoch from saved optimizer and RNG; source retained\'))\\n    cfg = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\\n    cfg.update(training_loss=\'explicit_aligned_seq2seq_ce_v1\', manifest=m, evaluation_sha256=fingerprint, dataset_sha256=data_hash.hexdigest(), resolved_model_commit=getattr(model.config, \'_commit_hash\', None), python=sys.version, versions={x: importlib.metadata.version(x) for x in [\'torch\',\'transformers\',\'Pillow\',\'numpy\']}, normalization=\'strip only; case/punctuation retained; WER uses whitespace split\', selection=\'minimum overall validation CER\', sampling=\'natural without replacement; balanced/de70 weighted replacement; N draws per epoch\')\\n    dump(args.out/\'config.json\', cfg)\\n    dump(args.out/\'evaluation.json\', evaluation)\\n\\n    class Dataset(torch.utils.data.Dataset):\\n        def __len__(self):\\n            return len(m[\'train\'])\\n        def __getitem__(self, i):\\n            r = m[\'train\'][i]\\n            with Image.open(r[\'image\']) as im:\\n                pixels = processor(images=im.convert(\'RGB\'), return_tensors=\'pt\').pixel_values[0]\\n            ids = labels[r[\'image\']]\\n            label = torch.tensor(ids + [tok.pad_token_id]*(args.max_length-len(ids)))\\n            label[label == tok.pad_token_id] = -100\\n            return pixels, label\\n\\n    @torch.no_grad()\\n    def evaluate(rows, name):\\n        model.eval()\\n        predictions = []\\n        for r in rows:\\n            with Image.open(r[\'image\']) as im:\\n                pixels = processor(images=im.convert(\'RGB\'), return_tensors=\'pt\').pixel_values.to(device)\\n            ids = model.generate(pixels, max_new_tokens=args.max_length, num_beams=1, do_sample=False)\\n            ref = Path(r[\'text\']).read_text(encoding=\'utf-8\').strip()\\n            pred = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()\\n            predictions.append(dict(**r, reference=ref, prediction=pred, character_errors=distance(ref,pred), word_errors=distance(ref.split(),pred.split()), reference_length=len(ref), prediction_length=len(pred), cer=distance(ref,pred)/len(ref), wer=distance(ref.split(),pred.split())/len(ref.split())))\\n        with (args.out/(name+\'.jsonl\')).open(\'w\', encoding=\'utf-8\') as f:\\n            for r in predictions:\\n                f.write(json.dumps(r, ensure_ascii=False)+\'\\\\n\')\\n        return metrics(predictions)\\n\\n    history = prior_history.copy()\\n    best_record = min(history,key=lambda r:r[\'validation\'][\'overall\'][\'cer\']) if history else None\\n    best = best_record[\'validation\'][\'overall\'][\'cer\'] if best_record else float(\'inf\')\\n    best_epoch = best_record[\'epoch\'] if best_record else None\\n    if not args.evaluate_only:\\n        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)\\n        if resume_state:\\n            optimizer.load_state_dict(resume_state[\'optimizer\'])\\n            torch.set_rng_state(resume_state[\'torch_rng\'])\\n            torch.cuda.set_rng_state_all(resume_state[\'cuda_rng\'])\\n            random.setstate(resume_state[\'python_rng\'])\\n            np.random.set_state(resume_state[\'numpy_rng\'])\\n            print(f\\"RESUMING AFTER COMPLETED EPOCH {resume_state[\'epoch\']}\\",flush=True)\\n        for epoch in range((resume_state[\'epoch\']+1) if resume_state else 1, args.epochs+1):\\n            generator = torch.Generator().manual_seed(seed+epoch)\\n            n = len(m[\'train\'])\\n            target = m[\'target_de_probability\']\\n            if target is None:\\n                draws = torch.randperm(n, generator=generator).tolist()\\n            else:\\n                counts = m[\'train_counts\']\\n                weights = torch.tensor([target/counts[\'de\'] if r[\'language\']==\'de\' else (1-target)/counts[\'la\'] for r in m[\'train\']], dtype=torch.double)\\n                draws = torch.multinomial(weights,n,replacement=True,generator=generator).tolist()\\n            dump(args.out/f\'draws_epoch{epoch}.json\', draws)\\n            loader = torch.utils.data.DataLoader(Dataset(),batch_size=args.batch_size,sampler=draws,num_workers=0)\\n            model.train()\\n            total, seen = 0., 0\\n            started = time.time()\\n            for step, (pixels, label) in enumerate(loader, 1):\\n                optimizer.zero_grad()\\n                target_labels = label.to(device)\\n                outputs = model(pixel_values=pixels.to(device), labels=target_labels)\\n                loss = aligned_training_loss(outputs.logits, target_labels)\\n                if not torch.isfinite(loss):\\n                    raise RuntimeError(\'Nonfinite loss\')\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\\n                optimizer.step()\\n                total += loss.item()*len(label)\\n                seen += len(label)\\n                if step % 100 == 0:\\n                    print(f\'epoch={epoch} step={step}/{len(loader)} loss={total/seen:.5f}\',flush=True)\\n            val = evaluate(evaluation[\'validation\'],f\'validation_epoch{epoch}\')\\n            record = dict(epoch=epoch, loss=total/seen, seconds=time.time()-started, validation=val, unique_draws=len(set(draws)), de_draws=sum(m[\'train\'][i][\'language\']==\'de\' for i in draws))\\n            history.append(record)\\n            # History is published only after all epoch files are saved.\\n            with (args.out/\'log.jsonl\').open(\'a\') as f:\\n                f.write(json.dumps(record)+\'\\\\n\')\\n            print(json.dumps(record),flush=True)\\n            if val[\'overall\'][\'cer\'] < best:\\n                best = val[\'overall\'][\'cer\']\\n                best_epoch = epoch\\n                # Selection is published after the complete epoch checkpoint.\\n            # Full last state, including RNG, for diagnosis/recovery; no implicit resume.\\n            checkpoint=args.out/f\'checkpoint_epoch{epoch}\'\\n            checkpoint.mkdir()\\n            model.save_pretrained(checkpoint)\\n            processor.save_pretrained(checkpoint)\\n            torch.save(dict(epoch=epoch,optimizer=optimizer.state_dict(),torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],python_rng=random.getstate(),numpy_rng=np.random.get_state()),checkpoint/\'training_state.pt\')\\n            dump(checkpoint/\'complete.json\',dict(epoch=epoch,best_epoch=best_epoch,best_validation_cer=best))\\n            dump(args.out/\'history.json\',history)\\n        del model\\n        if device.type == \'cuda\':\\n            torch.cuda.empty_cache()\\n        best_path=args.out/f\'checkpoint_epoch{best_epoch}\'\\n        if not best_path.exists(): best_path=args.resume_from/f\'checkpoint_epoch{best_epoch}\'\\n        model = VisionEncoderDecoderModel.from_pretrained(best_path).to(device)\\n    test = evaluate(evaluation[\'test\'],\'test_predictions\')\\n    dump(args.out/\'results.json\',dict(size=m[\'size\'],strategy=m[\'strategy\'],seed=seed,evaluation_sha256=fingerprint,best_epoch=best_epoch,best_validation_cer=best if history else None,test=test,evaluate_only=args.evaluate_only))\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n")\nsubprocess.run([sys.executable,\'-m\',\'py_compile\',str(script)],check=True)\nsys.path.insert(0,str(work))\nfrom prepare import audit\nnatural=json.loads((plan/\'n10000_natural_s42.json\').read_text());balanced=json.loads((plan/\'n10000_balanced_s42.json\').read_text())\nassert natural[\'train\']==balanced[\'train\'] and natural[\'seed\']==balanced[\'seed\']==42\nassert natural[\'target_de_probability\'] is None and balanced[\'target_de_probability\']==.5\ncfg=json.loads((root/\'runs_scale_weights_restart_v1/n10000_natural_s42_20261011T212741Z/config.json\').read_text())\nevaluation=json.loads((plan/\'evaluation.json\').read_text())\nimport hashlib\nassert hashlib.sha256(json.dumps(evaluation,sort_keys=True).encode()).hexdigest()==cfg[\'evaluation_sha256\']==natural[\'evaluation_sha256\']==balanced[\'evaluation_sha256\']\nassert not any(v for g in audit(dict(train=natural[\'train\'],**evaluation)).values() for v in g.values())\nneeded={str(Path(r[k]).relative_to(work)) if Path(r[k]).is_absolute() else str(Path(r[k])) for rows in [natural[\'train\'],*evaluation.values()] for r in rows for k in [\'image\',\'text\']}\nmissing={n for n in needed if not (work/n).is_file()}\nprint(\'CONTROL VERIFIED: SAME 10000 UNIQUE LINES; FIXED EVALUATION; MISSING FILES\',len(missing),flush=True)\nfor name in [\'dataset-baseline.zip\',\'dataset-additional.zip\']:\n with zipfile.ZipFile(root/\'input\'/name) as z:\n  chosen=[i for i in z.infolist() if i.filename in missing]\n  assert all((work/i.filename).resolve().is_relative_to(work.resolve()) for i in chosen)\n  assert shutil.disk_usage(work).free>sum(i.file_size for i in chosen)+4*1024**3\n  for j,i in enumerate(chosen,1):\n   z.extract(i,work)\n   if j%1000==0:print(\'RESTORING\',name,j,\'/\',len(chosen),flush=True)\n  missing.difference_update(i.filename for i in chosen)\nassert not missing\ncandidates=[p for p in root.glob(\'runs_sampling_control_v1/*/n10000_natural_s42\') if list(p.glob(\'checkpoint_epoch*/complete.json\'))]\nresume_parent=max(candidates,key=lambda p:max(int(q.parent.name.replace(\'checkpoint_epoch\',\'\')) for q in p.glob(\'checkpoint_epoch*/complete.json\'))) if candidates else None\nprint(\'RESUME NATURAL FROM\',resume_parent,flush=True)\nparent=root/\'runs_sampling_l4_complete_v1\'/time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime());parent.mkdir(parents=True)\n(parent/\'protocol.json\').write_text(json.dumps(dict(arms=[\'natural\',\'balanced\'],seed=42,initialization=\'same pretrained revision; no warm start or optimizer reset\',checkpoint=\'independent completed epoch directories\',evaluation_sha256=natural[\'evaluation_sha256\'],script_sha256=hashlib.sha256(script.read_bytes()).hexdigest(),settings={k:cfg[k] for k in [\'model\',\'revision\',\'epochs\',\'batch_size\',\'lr\',\'max_length\']}),indent=2))\nfor strategy in [\'natural\',\'balanced\']:\n out=parent/(\'n10000_\'+strategy+\'_s42\')\n cmd=[sys.executable,\'-u\',str(script),\'--manifest\',str(plan/(\'n10000_\'+strategy+\'_s42.json\')),\'--out\',str(out)]\n if strategy==\'natural\' and resume_parent:cmd.extend([\'--resume-from\',str(resume_parent)])\n for k in [\'model\',\'revision\',\'epochs\',\'batch_size\',\'lr\',\'max_length\']:cmd.extend([\'--\'+k.replace(\'_\',\'-\'),str(cfg[k])])\n print(\'STARTING CONTROL ARM\',strategy,out,flush=True)\n with subprocess.Popen(cmd,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as job:\n  for line in job.stdout:print(line,end=\'\',flush=True)\n  assert job.wait()==0,\'Arm failed; inspect epoch checkpoints before resuming\'\n print(\'COMPLETED ARM\',strategy,(out/\'results.json\').read_text(),flush=True)\nprint(\'BOTH SAMPLING ARMS COMPLETE\',parent,flush=True)\n', 'train_20k.py': 'from pathlib import Path\nimport json,zipfile,subprocess,sys,time,os,shutil\nroot=Path(\'/content/drive/MyDrive/BullingerAblation\');work=Path(\'/content/bullinger-ablation\');work.mkdir(exist_ok=True)\nsubprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'transformers==4.51.0\',\'Pillow==12.2.0\',\'numpy==2.2.6\'])\nplan=work/(\'sampling-control-plan-\'+time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime()));plan.mkdir()\nwith zipfile.ZipFile(root/\'input/experiment-code.zip\') as z:\n for n in z.namelist():\n  if n.startswith(\'plan/\') and n.endswith(\'.json\'):(plan/Path(n).name).write_bytes(z.read(n))\n (work/\'prepare.py\').write_bytes(z.read(\'prepare.py\'))\nscript=work/\'run_sampling_control.py\'\nscript.write_text("\\"\\"\\"Train one manifest, or evaluate a saved checkpoint on its fixed test set.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport random\\nimport shutil\\nimport sys\\nimport time\\nfrom prepare import audit, dump\\n\\n\\ndef distance(a, b):\\n    previous = list(range(len(b)+1))\\n    for i, x in enumerate(a, 1):\\n        current = [i]\\n        for j, y in enumerate(b, 1):\\n            current.append(min(current[-1]+1, previous[j]+1, previous[j-1]+(x != y)))\\n        previous = current\\n    return previous[-1]\\n\\n\\ndef metrics(rows):\\n    out = {}\\n    for group in [\'overall\', \'de\', \'la\']:\\n        subset = [r for r in rows if group == \'overall\' or r[\'language\'] == group]\\n        chars = sum(len(r[\'reference\']) for r in subset)\\n        words = sum(len(r[\'reference\'].split()) for r in subset)\\n        ce = sum(distance(r[\'reference\'], r[\'prediction\']) for r in subset)\\n        we = sum(distance(r[\'reference\'].split(), r[\'prediction\'].split()) for r in subset)\\n        out[group] = dict(n=len(subset), character_errors=ce, reference_chars=chars, word_errors=we, reference_words=words, cer=ce/chars if chars else None, wer=we/words if words else None)\\n    return out\\n\\n\\ndef aligned_training_loss(logits, labels):\\n    \\"\\"\\"Decoder inputs are already shifted by VisionEncoderDecoderModel.\\"\\"\\"\\n    import torch.nn.functional as F\\n    return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),\\n                           labels.to(logits.device).reshape(-1), ignore_index=-100)\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\'--manifest\', type=Path, required=True)\\n    p.add_argument(\'--out\', type=Path, required=True)\\n    p.add_argument(\'--model\', default=\'microsoft/trocr-base-handwritten\')\\n    p.add_argument(\'--revision\', default=\'main\', help=\'Use immutable Hugging Face commit for cross-machine reproduction\')\\n    p.add_argument(\'--evaluate-only\', action=\'store_true\')\\n    p.add_argument(\'--device\', choices=[\'cuda\', \'mps\', \'cpu\'], default=\'cuda\')\\n    p.add_argument(\'--epochs\', type=int, default=5)\\n    p.add_argument(\'--batch-size\', type=int, default=2)\\n    p.add_argument(\'--resume-from\', type=Path)\\n    p.add_argument(\'--lr\', type=float, default=1e-5)\\n    p.add_argument(\'--max-length\', type=int, default=128)\\n    args = p.parse_args()\\n    if args.epochs < 1 or args.batch_size < 1 or args.max_length < 2 or args.lr <= 0:\\n        raise ValueError(\'Invalid training parameters\')\\n    if args.out.exists():\\n        raise ValueError(\'Output exists; choose a fresh run directory\')\\n    m = json.loads(args.manifest.read_text())\\n    ep = args.manifest.parent / m[\'evaluation_file\']\\n    evaluation = json.loads(ep.read_text())\\n    fingerprint = hashlib.sha256(json.dumps(evaluation, sort_keys=True).encode()).hexdigest()\\n    if fingerprint != m[\'evaluation_sha256\']:\\n        raise ValueError(\'Evaluation manifest changed\')\\n    split = dict(train=m[\'train\'], **evaluation)\\n    if any(v for d in audit(split).values() for v in d.values()):\\n        raise ValueError(\'Letter or line leakage\')\\n    for rows in split.values():\\n        for r in rows:\\n            if r[\'language\'] not in [\'de\', \'la\'] or not Path(r[\'image\']).is_file() or not Path(r[\'text\']).is_file():\\n                raise ValueError(f\'Invalid sample {r}\')\\n    os.environ[\'CUBLAS_WORKSPACE_CONFIG\'] = \':4096:8\'\\n    import numpy as np\\n    import torch\\n    from PIL import Image\\n    from transformers import TrOCRProcessor, VisionEncoderDecoderModel\\n    seed = m[\'seed\']\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.use_deterministic_algorithms(True)\\n    device = torch.device(args.device)\\n    if args.device == \'cuda\' and not torch.cuda.is_available():\\n        raise RuntimeError(\'CUDA unavailable; use an explicit --device cpu/mps if desired\')\\n    completed_checkpoints = sorted(args.resume_from.glob(\'checkpoint_epoch*/complete.json\'),key=lambda p:int(p.parent.name.replace(\'checkpoint_epoch\',\'\'))) if args.resume_from else []\\n    if args.resume_from and not completed_checkpoints: raise ValueError(\'No complete epoch checkpoints\')\\n    checkpoint = completed_checkpoints[-1].parent if completed_checkpoints else None\\n    source_model = checkpoint if args.resume_from else args.model\\n    load_revision = None if args.resume_from else args.revision\\n    processor = TrOCRProcessor.from_pretrained(source_model, revision=load_revision, use_fast=False)\\n    model = VisionEncoderDecoderModel.from_pretrained(source_model, revision=load_revision).to(device)\\n    tok = processor.tokenizer\\n    for config in [model.config, model.generation_config]:\\n        config.decoder_start_token_id = tok.cls_token_id\\n        config.pad_token_id = tok.pad_token_id\\n        config.eos_token_id = tok.sep_token_id\\n    labels = {}\\n    data_hash = hashlib.sha256()\\n    for rows in split.values():\\n        for r in rows:\\n            text = Path(r[\'text\']).read_text(encoding=\'utf-8\').strip()\\n            if not text:\\n                raise ValueError(\'Empty reference\')\\n            ids = tok(text).input_ids\\n            if len(ids) > args.max_length:\\n                raise ValueError(f\'Label would truncate ({len(ids)} tokens): {r[\\"text\\"]}; increase --max-length for ALL runs\')\\n            labels[r[\'image\']] = ids\\n            data_hash.update(Path(r[\'image\']).read_bytes())\\n            data_hash.update(text.encode())\\n    resume_state = None\\n    prior_history = []\\n    if args.resume_from:\\n        prior_cfg = json.loads((args.resume_from/\'config.json\').read_text())\\n        if prior_cfg.get(\'training_loss\') != \'explicit_aligned_seq2seq_ce_v1\':\\n            raise ValueError(\'Cannot resume misaligned training\')\\n        for key, value in dict(model=args.model, revision=args.revision, max_length=args.max_length, epochs=args.epochs, batch_size=args.batch_size, lr=args.lr, evaluation_sha256=fingerprint, dataset_sha256=data_hash.hexdigest(), manifest=m).items():\\n            if prior_cfg[key] != value:\\n                raise ValueError(f\'Resume protocol mismatch: {key}\')\\n        if prior_cfg[\'versions\'] != {x:importlib.metadata.version(x) for x in [\'torch\',\'transformers\',\'Pillow\',\'numpy\']}:\\n            raise ValueError(\'Resume library versions differ\')\\n        resume_state = torch.load(checkpoint/\'training_state.pt\', map_location=\'cpu\', weights_only=False)\\n        prior_history = [r for r in json.loads((args.resume_from/\'history.json\').read_text()) if r[\'epoch\']<=resume_state[\'epoch\']]\\n        completed = resume_state[\'epoch\']\\n        if [r[\'epoch\'] for r in prior_history] != list(range(1,completed+1)):\\n            raise ValueError(\'Checkpoint and history disagree; do not silently resume\')\\n    args.out.mkdir(parents=True)\\n    if args.resume_from:\\n        # Completed parent checkpoints stay read-only.\\n        dump(args.out/\'history.json\',prior_history)\\n        for r in prior_history:\\n            epoch=r[\'epoch\']\\n            for filename in [f\'draws_epoch{epoch}.json\',f\'validation_epoch{epoch}.jsonl\']:\\n                src=args.resume_from/filename\\n                if src.exists(): shutil.copy2(src,args.out/filename)\\n        dump(args.out/\'resume_provenance.json\',dict(parent=str(args.resume_from),completed_epoch=resume_state[\'epoch\'],replay=\'Restart incomplete next epoch from saved optimizer and RNG; source retained\'))\\n    cfg = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\\n    cfg.update(training_loss=\'explicit_aligned_seq2seq_ce_v1\', manifest=m, evaluation_sha256=fingerprint, dataset_sha256=data_hash.hexdigest(), resolved_model_commit=getattr(model.config, \'_commit_hash\', None), python=sys.version, versions={x: importlib.metadata.version(x) for x in [\'torch\',\'transformers\',\'Pillow\',\'numpy\']}, normalization=\'strip only; case/punctuation retained; WER uses whitespace split\', selection=\'minimum overall validation CER\', sampling=\'natural without replacement; balanced/de70 weighted replacement; N draws per epoch\')\\n    dump(args.out/\'config.json\', cfg)\\n    dump(args.out/\'evaluation.json\', evaluation)\\n\\n    class Dataset(torch.utils.data.Dataset):\\n        def __len__(self):\\n            return len(m[\'train\'])\\n        def __getitem__(self, i):\\n            r = m[\'train\'][i]\\n            with Image.open(r[\'image\']) as im:\\n                pixels = processor(images=im.convert(\'RGB\'), return_tensors=\'pt\').pixel_values[0]\\n            ids = labels[r[\'image\']]\\n            label = torch.tensor(ids + [tok.pad_token_id]*(args.max_length-len(ids)))\\n            label[label == tok.pad_token_id] = -100\\n            return pixels, label\\n\\n    @torch.no_grad()\\n    def evaluate(rows, name):\\n        model.eval()\\n        predictions = []\\n        for r in rows:\\n            with Image.open(r[\'image\']) as im:\\n                pixels = processor(images=im.convert(\'RGB\'), return_tensors=\'pt\').pixel_values.to(device)\\n            ids = model.generate(pixels, max_new_tokens=args.max_length, num_beams=1, do_sample=False)\\n            ref = Path(r[\'text\']).read_text(encoding=\'utf-8\').strip()\\n            pred = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()\\n            predictions.append(dict(**r, reference=ref, prediction=pred, character_errors=distance(ref,pred), word_errors=distance(ref.split(),pred.split()), reference_length=len(ref), prediction_length=len(pred), cer=distance(ref,pred)/len(ref), wer=distance(ref.split(),pred.split())/len(ref.split())))\\n        with (args.out/(name+\'.jsonl\')).open(\'w\', encoding=\'utf-8\') as f:\\n            for r in predictions:\\n                f.write(json.dumps(r, ensure_ascii=False)+\'\\\\n\')\\n        return metrics(predictions)\\n\\n    history = prior_history.copy()\\n    best_record = min(history,key=lambda r:r[\'validation\'][\'overall\'][\'cer\']) if history else None\\n    best = best_record[\'validation\'][\'overall\'][\'cer\'] if best_record else float(\'inf\')\\n    best_epoch = best_record[\'epoch\'] if best_record else None\\n    if not args.evaluate_only:\\n        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)\\n        if resume_state:\\n            optimizer.load_state_dict(resume_state[\'optimizer\'])\\n            torch.set_rng_state(resume_state[\'torch_rng\'])\\n            torch.cuda.set_rng_state_all(resume_state[\'cuda_rng\'])\\n            random.setstate(resume_state[\'python_rng\'])\\n            np.random.set_state(resume_state[\'numpy_rng\'])\\n            print(f\\"RESUMING AFTER COMPLETED EPOCH {resume_state[\'epoch\']}\\",flush=True)\\n        for epoch in range((resume_state[\'epoch\']+1) if resume_state else 1, args.epochs+1):\\n            generator = torch.Generator().manual_seed(seed+epoch)\\n            n = len(m[\'train\'])\\n            target = m[\'target_de_probability\']\\n            if target is None:\\n                draws = torch.randperm(n, generator=generator).tolist()\\n            else:\\n                counts = m[\'train_counts\']\\n                weights = torch.tensor([target/counts[\'de\'] if r[\'language\']==\'de\' else (1-target)/counts[\'la\'] for r in m[\'train\']], dtype=torch.double)\\n                draws = torch.multinomial(weights,n,replacement=True,generator=generator).tolist()\\n            dump(args.out/f\'draws_epoch{epoch}.json\', draws)\\n            loader = torch.utils.data.DataLoader(Dataset(),batch_size=args.batch_size,sampler=draws,num_workers=0)\\n            model.train()\\n            total, seen = 0., 0\\n            started = time.time()\\n            for step, (pixels, label) in enumerate(loader, 1):\\n                optimizer.zero_grad()\\n                target_labels = label.to(device)\\n                outputs = model(pixel_values=pixels.to(device), labels=target_labels)\\n                loss = aligned_training_loss(outputs.logits, target_labels)\\n                if not torch.isfinite(loss):\\n                    raise RuntimeError(\'Nonfinite loss\')\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\\n                optimizer.step()\\n                total += loss.item()*len(label)\\n                seen += len(label)\\n                if step % 100 == 0:\\n                    print(f\'epoch={epoch} step={step}/{len(loader)} loss={total/seen:.5f}\',flush=True)\\n            val = evaluate(evaluation[\'validation\'],f\'validation_epoch{epoch}\')\\n            record = dict(epoch=epoch, loss=total/seen, seconds=time.time()-started, validation=val, unique_draws=len(set(draws)), de_draws=sum(m[\'train\'][i][\'language\']==\'de\' for i in draws))\\n            history.append(record)\\n            # History is published only after all epoch files are saved.\\n            with (args.out/\'log.jsonl\').open(\'a\') as f:\\n                f.write(json.dumps(record)+\'\\\\n\')\\n            print(json.dumps(record),flush=True)\\n            if val[\'overall\'][\'cer\'] < best:\\n                best = val[\'overall\'][\'cer\']\\n                best_epoch = epoch\\n                # Selection is published after the complete epoch checkpoint.\\n            # Full last state, including RNG, for diagnosis/recovery; no implicit resume.\\n            checkpoint=args.out/f\'checkpoint_epoch{epoch}\'\\n            checkpoint.mkdir()\\n            model.save_pretrained(checkpoint)\\n            processor.save_pretrained(checkpoint)\\n            torch.save(dict(epoch=epoch,optimizer=optimizer.state_dict(),torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],python_rng=random.getstate(),numpy_rng=np.random.get_state()),checkpoint/\'training_state.pt\')\\n            dump(checkpoint/\'complete.json\',dict(epoch=epoch,best_epoch=best_epoch,best_validation_cer=best))\\n            dump(args.out/\'history.json\',history)\\n        del model\\n        if device.type == \'cuda\':\\n            torch.cuda.empty_cache()\\n        best_path=args.out/f\'checkpoint_epoch{best_epoch}\'\\n        if not best_path.exists(): best_path=args.resume_from/f\'checkpoint_epoch{best_epoch}\'\\n        model = VisionEncoderDecoderModel.from_pretrained(best_path).to(device)\\n    test = evaluate(evaluation[\'test\'],\'test_predictions\')\\n    dump(args.out/\'results.json\',dict(size=m[\'size\'],strategy=m[\'strategy\'],seed=seed,evaluation_sha256=fingerprint,best_epoch=best_epoch,best_validation_cer=best if history else None,test=test,evaluate_only=args.evaluate_only))\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n")\nsubprocess.run([sys.executable,\'-m\',\'py_compile\',str(script)],check=True)\nsys.path.insert(0,str(work))\nfrom prepare import audit\nnatural=json.loads((plan/\'n20000_natural_s42.json\').read_text());balanced=json.loads((plan/\'n20000_balanced_s42.json\').read_text())\nassert natural[\'train\']==balanced[\'train\'] and natural[\'seed\']==balanced[\'seed\']==42\nassert natural[\'target_de_probability\'] is None and balanced[\'target_de_probability\']==.5\ncfg=json.loads((root/\'runs_scale_weights_restart_v1/n10000_natural_s42_20261011T212741Z/config.json\').read_text())\nevaluation=json.loads((plan/\'evaluation.json\').read_text())\nimport hashlib\nassert hashlib.sha256(json.dumps(evaluation,sort_keys=True).encode()).hexdigest()==cfg[\'evaluation_sha256\']==natural[\'evaluation_sha256\']==balanced[\'evaluation_sha256\']\nassert not any(v for g in audit(dict(train=natural[\'train\'],**evaluation)).values() for v in g.values())\nassert len(natural[\'train\'])==20000\nprint(\'20K MANIFEST COUNTS\',natural[\'train_counts\'],flush=True)\nneeded={str(Path(r[k]).relative_to(work)) if Path(r[k]).is_absolute() else str(Path(r[k])) for rows in [natural[\'train\'],*evaluation.values()] for r in rows for k in [\'image\',\'text\']}\nmissing={n for n in needed if not (work/n).is_file()}\nprint(\'CONTROL VERIFIED: SAME 20000 UNIQUE LINES; FIXED EVALUATION; MISSING FILES\',len(missing),flush=True)\nfor name in [\'dataset-baseline.zip\',\'dataset-additional.zip\']:\n with zipfile.ZipFile(root/\'input\'/name) as z:\n  chosen=[i for i in z.infolist() if i.filename in missing]\n  assert all((work/i.filename).resolve().is_relative_to(work.resolve()) for i in chosen)\n  assert shutil.disk_usage(work).free>sum(i.file_size for i in chosen)+4*1024**3\n  for j,i in enumerate(chosen,1):\n   z.extract(i,work)\n   if j%1000==0:print(\'RESTORING\',name,j,\'/\',len(chosen),flush=True)\n  missing.difference_update(i.filename for i in chosen)\nassert not missing\nparent=root/\'runs_scale20k_v1\'/time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime());parent.mkdir(parents=True)\n\n# Rank validation errors only; targets and original files remain unchanged.\nimport csv\nprevious=root/\'runs_sampling_l4_complete_v1/20261012T142349Z\'\nreviews=sorted(previous.glob(\'beam_validation_*/beam4_predictions.jsonl\'))\nif reviews:\n from collections import Counter\n rows=[json.loads(x) for x in reviews[-1].read_text().splitlines()]\n def substitutions(a,b):\n  dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]\n  for i in range(len(a)+1):dp[i][0]=i\n  for j in range(len(b)+1):dp[0][j]=j\n  for i in range(1,len(a)+1):\n   for j in range(1,len(b)+1):dp[i][j]=min(dp[i-1][j]+1,dp[i][j-1]+1,dp[i-1][j-1]+(a[i-1]!=b[j-1]))\n  i,j=len(a),len(b);s=0\n  while i or j:\n   if i and j and dp[i][j]==dp[i-1][j-1]+(a[i-1]!=b[j-1]):s+=a[i-1]!=b[j-1];i-=1;j-=1\n   elif i and dp[i][j]==dp[i-1][j]+1:i-=1\n   else:j-=1\n  return s,dp[-1][-1]\n ranked=[]\n for r in rows:\n  sub,edits=substitutions(r[\'reference\'].split(),r[\'prediction\'].split())\n  ranked.append(dict(language=r[\'language\'],image=r[\'image\'],reference=r[\'reference\'],prediction=r[\'prediction\'],word_substitutions=sub,word_edits=edits,review_status=\'unreviewed; do not rewrite labels automatically\'))\n selected=[]\n for language in [\'de\',\'la\']:\n  selected.extend(sorted([r for r in ranked if r[\'language\']==language],key=lambda r:(r[\'word_substitutions\'],r[\'word_edits\']),reverse=True)[:20])\n path=parent/\'validation_high_substitution_review.csv\'\n with path.open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n  writer=csv.DictWriter(f,fieldnames=list(selected[0]));writer.writeheader();writer.writerows(selected)\n print(\'VALIDATION REVIEW QUEUE SAVED\',path,\'ROWS\',len(selected),flush=True)\n\n(parent/\'protocol.json\').write_text(json.dumps(dict(arms=[\'natural\'],seed=42,initialization=\'same pretrained revision; no warm start or optimizer reset\',checkpoint=\'independent completed epoch directories\',evaluation_sha256=natural[\'evaluation_sha256\'],script_sha256=hashlib.sha256(script.read_bytes()).hexdigest(),settings={k:cfg[k] for k in [\'model\',\'revision\',\'epochs\',\'batch_size\',\'lr\',\'max_length\']}),indent=2))\nfor strategy in [\'natural\']:\n out=parent/(\'n20000_\'+strategy+\'_s42\')\n cmd=[sys.executable,\'-u\',str(script),\'--manifest\',str(plan/(\'n20000_\'+strategy+\'_s42.json\')),\'--out\',str(out)]\n for k in [\'model\',\'revision\',\'epochs\',\'batch_size\',\'lr\',\'max_length\']:cmd.extend([\'--\'+k.replace(\'_\',\'-\'),str(cfg[k])])\n print(\'STARTING CONTROL ARM\',strategy,out,flush=True)\n with subprocess.Popen(cmd,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as job:\n  for line in job.stdout:print(line,end=\'\',flush=True)\n  assert job.wait()==0,\'Arm failed; inspect epoch checkpoints before resuming\'\n print(\'COMPLETED ARM\',strategy,(out/\'results.json\').read_text(),flush=True)\nprint(\'20K NATURAL COMPLETE\',parent,flush=True)\n', 'adapt_validation.py': '"""Three-arm 500-step validation-only adaptation; original references retained."""\nfrom pathlib import Path\nimport json,sys,subprocess,time,zipfile,hashlib,random,shutil\nROOT=Path(\'/content/drive/MyDrive/BullingerAblation\');WORK=Path(\'/content/adapt20k\');WORK.mkdir(exist_ok=True)\nsubprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'transformers==4.51.0\',\'Pillow==12.2.0\',\'numpy==2.2.6\'])\nBASE=ROOT/\'runs_scale20k_v1/20261012T195316Z/n20000_natural_s42\'\ncfg=json.loads((BASE/\'config.json\').read_text());m=cfg[\'manifest\'];ev=json.loads((BASE/\'evaluation.json\').read_text())\nassert len(m[\'train\'])==20000\nassert hashlib.sha256(json.dumps(ev,sort_keys=True).encode()).hexdigest()==cfg[\'evaluation_sha256\']\nassert json.loads((BASE/\'results.json\').read_text())[\'best_epoch\']==5\nselections=sorted(BASE.parent.glob(\'beam_validation_*/selection.json\'));selection=json.loads(selections[-1].read_text())\nassert selection[\'selected_beam\']==8 and selection[\'parent\']==str(BASE)\nletter=lambda r:Path(r[\'image\']).stem.split(\'_\')[0]\nassert not ({letter(r) for r in m[\'train\']}&{letter(r) for rows in ev.values() for r in rows})\nassert not ({letter(r) for r in ev[\'validation\']}&{letter(r) for r in ev[\'test\']})\nimport argparse\np=argparse.ArgumentParser();p.add_argument(\'--seed\',type=int,required=True);args=p.parse_args()\nseed=args.seed;budget=1000\nnatural=list(range(20000));random.Random(seed+1).shuffle(natural);natural=natural[:budget]\ngroups={l:[i for i,r in enumerate(m[\'train\']) if r[\'language\']==l] for l in [\'de\',\'la\']}\nfor l in groups:random.Random(seed+1).shuffle(groups[l])\nde20=groups[\'de\'][:200]+groups[\'la\'][:800];random.Random(seed+1).shuffle(de20)\nde30=groups[\'de\'][:300]+groups[\'la\'][:700];random.Random(seed+1).shuffle(de30)\narms={\'natural\':natural,\'de20\':de20};assert all(len(set(x))==budget for x in arms.values())\nout=ROOT/\'adapt20k_replication_v1\'/(\'seed\'+str(seed)+\'_\'+time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime()));out.mkdir(parents=True)\n(out/\'protocol.json\').write_text(json.dumps(dict(parent=str(BASE),source_checkpoint=5,seed=seed,steps=500,batch_size=2,learning_rate=2e-6,decode_beams=8,max_length=128,evaluation_sha256=cfg[\'evaluation_sha256\'],sampling=\'natural permutation vs exact 200DE/800LA without replacement from same 20k pool\',selection=\'retain baseline unless candidate improves at least one metric without worsening the other; compare target-normalized worst ratio among eligible candidates\',test_evaluated=False,labels_modified=False,worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()),indent=2))\nfor name,indices in arms.items():(out/(name+\'_indices.json\')).write_text(json.dumps(indices))\nprint(\'PAIRED PROTOCOL SAVED\',out,flush=True)\nneeded_rows=[m[\'train\'][i] for i in set(natural+de20)]+ev[\'validation\']\nneeded={str(Path(\'dataset\')/r[\'language\']/Path(r[k]).name) for r in needed_rows for k in [\'image\',\'text\']}\nfor archive in [\'dataset-baseline.zip\',\'dataset-additional.zip\']:\n with zipfile.ZipFile(ROOT/\'input\'/archive) as z:\n  chosen=[i for i in z.infolist() if i.filename in needed and not (WORK/i.filename).exists()]\n  assert shutil.disk_usage(WORK).free>sum(i.file_size for i in chosen)+3*1024**3\n  for j,i in enumerate(chosen,1):\n   assert (WORK/i.filename).resolve().is_relative_to(WORK.resolve());z.extract(i,WORK)\n   if j%1000==0:print(\'RESTORED\',archive,j,\'/\',len(chosen),flush=True)\nassert all((WORK/n).is_file() for n in needed)\nimport os\nos.environ[\'CUBLAS_WORKSPACE_CONFIG\']=\':4096:8\'\nimport numpy as np,torch\nfrom PIL import Image\nfrom transformers import TrOCRProcessor,VisionEncoderDecoderModel\nassert torch.cuda.is_available();print(\'GPU\',torch.cuda.get_device_name(0),flush=True)\ntorch.use_deterministic_algorithms(True)\nparent_adapt=ROOT/\'adapt20k_paired_v1/20261013T101050Z\'\nassert json.loads((parent_adapt/\'selection.json\').read_text())[\'selected_arm\']==\'de20\'\nsource=parent_adapt/\'de20\'\n(out/\'parent.json\').write_text(json.dumps(dict(weights=str(source),optimizer=\'fresh AdamW in each arm\',seed=seed),indent=2));local_model=WORK/\'initial_model\'\nif not local_model.exists():shutil.copytree(source,local_model,ignore=shutil.ignore_patterns(\'training_state.pt\'),dirs_exist_ok=False)\nprocessor=TrOCRProcessor.from_pretrained(local_model,use_fast=False);tok=processor.tokenizer\npath=lambda r,k:WORK/\'dataset\'/r[\'language\']/Path(r[k]).name\nlabels={}\nfor r in needed_rows:\n text=path(r,\'text\').read_text(encoding=\'utf-8\').strip();ids=tok(text).input_ids\n assert text and len(ids)<=128\n labels[r[\'image\']]=ids\n\ndef distance(a,b):\n prev=list(range(len(b)+1))\n for i,x in enumerate(a,1):\n  cur=[i]\n  for j,y in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n  prev=cur\n return prev[-1]\ndef metrics(rows):\n result={}\n for group in [\'overall\',\'de\',\'la\']:\n  rs=[r for r in rows if group==\'overall\' or r[\'language\']==group];chars=sum(len(r[\'reference\']) for r in rs);words=sum(len(r[\'reference\'].split()) for r in rs)\n  ce=sum(distance(r[\'reference\'],r[\'prediction\']) for r in rs);we=sum(distance(r[\'reference\'].split(),r[\'prediction\'].split()) for r in rs)\n  result[group]=dict(n=len(rs),character_errors=ce,reference_chars=chars,word_errors=we,reference_words=words,cer=ce/chars if chars else None,wer=we/words if words else None)\n return result\n@torch.inference_mode()\ndef evaluate(model,name):\n model.eval();rows=[]\n for i,r in enumerate(ev[\'validation\'],1):\n  with Image.open(path(r,\'image\')) as im:x=processor(images=im.convert(\'RGB\'),return_tensors=\'pt\').pixel_values.to(\'cuda\')\n  ids=model.generate(x,max_new_tokens=128,num_beams=8,do_sample=False,use_cache=False)\n  rows.append(dict(line_id=Path(r[\'image\']).stem,language=r[\'language\'],reference=path(r,\'text\').read_text(encoding=\'utf-8\').strip(),prediction=processor.batch_decode(ids,skip_special_tokens=True)[0].strip()))\n  if i%50==0:print(name,\'VALIDATION\',i,\'/246\',flush=True)\n (out/(name+\'_validation.jsonl\')).write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows))\n return metrics(rows)\ndef load_model():\n random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n model=VisionEncoderDecoderModel.from_pretrained(local_model).to(\'cuda\')\n for c in [model.config,model.generation_config]:c.decoder_start_token_id=tok.cls_token_id;c.pad_token_id=tok.pad_token_id;c.eos_token_id=tok.sep_token_id\n return model\nbaseline=None;records=[]\nfor name,indices in arms.items():\n model=load_model()\n if baseline is None:\n  baseline=evaluate(model,\'baseline\')\n  expected=json.loads((parent_adapt/\'de20/complete.json\').read_text())[\'validation\']\n  assert baseline==expected,\'Baseline does not reproduce frozen validation\'\n  print(\'BASELINE VERIFIED\',json.dumps(baseline),flush=True)\n optimizer=torch.optim.AdamW(model.parameters(),lr=2e-6);model.train()\n random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n total=0.;started=time.time()\n for step in range(500):\n  rs=[m[\'train\'][i] for i in indices[step*2:step*2+2]];images=[];ys=[]\n  for r in rs:\n   with Image.open(path(r,\'image\')) as im:images.append(im.convert(\'RGB\'))\n   ids=labels[r[\'image\']];ys.append(ids+[tok.pad_token_id]*(128-len(ids)))\n  x=processor(images=images,return_tensors=\'pt\').pixel_values.to(\'cuda\');y=torch.tensor(ys,device=\'cuda\');y[y==tok.pad_token_id]=-100\n  optimizer.zero_grad();outputs=model(pixel_values=x,labels=y)\n  loss=torch.nn.functional.cross_entropy(outputs.logits.float().reshape(-1,outputs.logits.shape[-1]),y.reshape(-1),ignore_index=-100)\n  assert torch.isfinite(loss);loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step();total+=loss.item()\n  if (step+1)%100==0:print(\'ADAPT\',name,\'STEP\',step+1,\'/500 LOSS\',total/(step+1),flush=True)\n val=evaluate(model,name);d=out/name;d.mkdir();model.save_pretrained(d);processor.save_pretrained(d)\n torch.save(dict(step=500,optimizer=optimizer.state_dict(),torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all(),python_rng=random.getstate(),numpy_rng=np.random.get_state()),d/\'training_state.pt\')\n b=baseline[\'overall\'];v=val[\'overall\'];eligible=v[\'cer\']<=b[\'cer\'] and v[\'wer\']<=b[\'wer\'] and (v[\'cer\']<b[\'cer\'] or v[\'wer\']<b[\'wer\'])\n record=dict(arm=name,validation=val,baseline=baseline,eligible=eligible,meets_targets=v[\'cer\']<=.15 and v[\'wer\']<.37,mean_loss=total/500,seconds=time.time()-started,de_draws=sum(m[\'train\'][i][\'language\']==\'de\' for i in indices),unique_draws=len(set(indices)),test_evaluated=False)\n (d/\'complete.json\').write_text(json.dumps(record,indent=2));records.append(record);(out/\'summary.json\').write_text(json.dumps(records,indent=2));print(\'ADAPT RESULT\',json.dumps(record),flush=True)\n del model,optimizer,outputs,loss;torch.cuda.empty_cache()\nchoices=[dict(arm=\'baseline\',validation=baseline)]+[r for r in records if r[\'eligible\']]\nbest=min(choices,key=lambda r:max(r[\'validation\'][\'overall\'][\'cer\']/.15,r[\'validation\'][\'overall\'][\'wer\']/.37))\n(out/\'selection.json\').write_text(json.dumps(dict(selected_arm=best[\'arm\'],criterion=\'baseline or joint non-worsening candidate; target-normalized worst ratio\',test_evaluated=False),indent=2))\nprint(\'PAIRED ADAPTATION COMPLETE; SELECTED\',best[\'arm\'],\'OUTPUT\',out,flush=True)\n', 'rerank_validation.py': '"""Train-text-only character/word LM reranking of beam8 validation candidates."""\nfrom pathlib import Path\nimport json,time,hashlib,zipfile,collections,math,subprocess,sys,shutil,os\nROOT=Path(\'/content/drive/MyDrive/BullingerAblation\');WORK=Path(\'/content/wer-rescoring\');WORK.mkdir(exist_ok=True)\nBASE=ROOT/\'runs_scale20k_v1/20261012T195316Z/n20000_natural_s42\'\nSOURCE=ROOT/\'adapt20k_replication_v1/seed44_20261013T172351Z\'\ncfg=json.loads((BASE/\'config.json\').read_text());train=cfg[\'manifest\'][\'train\'];ev=json.loads((BASE/\'evaluation.json\').read_text())\nassert len(train)==20000 and len(ev[\'validation\'])==246\nassert hashlib.sha256(json.dumps(ev,sort_keys=True).encode()).hexdigest()==cfg[\'evaluation_sha256\']\nletter=lambda r:Path(r[\'image\']).stem.split(\'_\')[0]\nassert not ({letter(r) for r in train}&{letter(r) for r in ev[\'validation\']})\n# Only training TXT and validation images are restored; no test rows or test references.\nneeded={str(Path(\'dataset\')/r[\'language\']/Path(r[\'text\']).name) for r in train}\nneeded.update(str(Path(\'dataset\')/r[\'language\']/Path(r[\'image\']).name) for r in ev[\'validation\'])\nfor name in [\'dataset-baseline.zip\',\'dataset-additional.zip\']:\n with zipfile.ZipFile(ROOT/\'input\'/name) as z:\n  for info in z.infolist():\n   if info.filename in needed:\n    assert (WORK/info.filename).resolve().is_relative_to(WORK.resolve())\n    z.extract(info,WORK);needed.remove(info.filename)\nassert not needed,\'Missing data\'\nOUT=ROOT/\'wer_rescoring_v1\'/time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime());OUT.mkdir(parents=True)\npath=lambda r,k:WORK/\'dataset\'/r[\'language\']/Path(r[k]).name\nwords={l:collections.Counter() for l in [\'de\',\'la\']};grams={l:collections.Counter() for l in [\'de\',\'la\']};contexts={l:collections.Counter() for l in [\'de\',\'la\']};alphabet={l:set() for l in [\'de\',\'la\']};digest=hashlib.sha256()\nfor r in train:\n text=path(r,\'text\').read_text(encoding=\'utf-8\').strip();l=r[\'language\'];digest.update(json.dumps([r[\'image\'],l,text],ensure_ascii=False).encode());words[l].update(text.split());alphabet[l].update(text+\'$\')\n s=\'^^\'+text+\'$\'\n for i in range(2,len(s)):grams[l][s[i-2:i+1]]+=1;contexts[l][s[i-2:i]]+=1\n# LM is fitted once, solely on the frozen 20k training references. No modern dictionaries.\ndef lm(text,l):\n s=\'^^\'+text+\'$\';v=len(alphabet[l])+1\n c=sum(math.log((grams[l][s[i-2:i+1]]+.1)/(contexts[l][s[i-2:i]]+.1*v)) for i in range(2,len(s)))/max(1,len(s)-2)\n w=text.split();n=sum(words[l].values());vocab=len(words[l])+1\n u=sum(math.log((words[l][token]+.1)/(n+.1*vocab)) for token in w)/max(1,len(w))\n return c,u\nweights=[(a,b) for a in [0.,.01,.03,.1] for b in [0.,.01,.03]]\nprotocol=dict(source=str(SOURCE/\'de20\'),seed=44,validation_lines=246,evaluation_sha256=cfg[\'evaluation_sha256\'],training_text_sha256=digest.hexdigest(),lm=\'case-sensitive character trigram + word unigram, add0.1 smoothing, average log probabilities\',candidate_generation=\'beam8, return8, original max128, no sample\',weights=weights,selection=\'overall raw CER/WER must both not worsen; minimize max(CER/.15,WER/.37); retain baseline on ties\',language_routing=\'manifest DE/LA; requires known language at inference\',test_evaluated=False,labels_modified=False,training_started=False)\n(OUT/\'protocol.json\').write_text(json.dumps(protocol,indent=2));print(\'RESCORING OUTPUT\',OUT,flush=True)\nsubprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'transformers==4.51.0\',\'Pillow==12.2.0\',\'numpy==2.2.6\'])\nos.environ[\'CUBLAS_WORKSPACE_CONFIG\']=\':4096:8\'\nimport torch\nfrom PIL import Image\nfrom transformers import TrOCRProcessor,VisionEncoderDecoderModel\ntorch.manual_seed(44);torch.use_deterministic_algorithms(True);assert torch.cuda.is_available();print(\'GPU\',torch.cuda.get_device_name(0),flush=True)\nlocal=WORK/\'model\';shutil.copytree(SOURCE/\'de20\',local,ignore=shutil.ignore_patterns(\'training_state.pt\'),dirs_exist_ok=False)\nprocessor=TrOCRProcessor.from_pretrained(local,use_fast=False);model=VisionEncoderDecoderModel.from_pretrained(local).to(\'cuda\').eval();tok=processor.tokenizer\nfor c in [model.config,model.generation_config]:c.decoder_start_token_id=tok.cls_token_id;c.pad_token_id=tok.pad_token_id;c.eos_token_id=tok.sep_token_id\nsaved=[json.loads(s) for s in (SOURCE/\'de20_validation.jsonl\').read_text().splitlines()];refs={r[\'line_id\']:r for r in saved};assert len(refs)==246\nrows=[]\nwith torch.inference_mode():\n for i,r in enumerate(ev[\'validation\'],1):\n  line=Path(r[\'image\']).stem;ref=refs[line];assert ref[\'language\']==r[\'language\']\n  with Image.open(path(r,\'image\')) as im:x=processor(images=im.convert(\'RGB\'),return_tensors=\'pt\').pixel_values.to(\'cuda\')\n  generated=model.generate(x,num_beams=8,num_return_sequences=8,max_new_tokens=128,do_sample=False,use_cache=False,return_dict_in_generate=True,output_scores=True)\n  texts=[s.strip() for s in processor.batch_decode(generated.sequences,skip_special_tokens=True)]\n  assert texts[0]==ref[\'prediction\'],\'Top beam differs from frozen baseline; stop comparison\'\n  candidates=[]\n  for text,score in zip(texts,generated.sequences_scores.cpu().tolist()):\n   char,word=lm(text,r[\'language\']);candidates.append(dict(text=text,model_score=score,char_score=char,word_score=word))\n  rows.append(dict(**ref,candidates=candidates))\n  if i%25==0:print(\'CANDIDATES\',i,\'/246\',flush=True)\n(OUT/\'candidates.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows))\ndef distance(a,b):\n prev=list(range(len(b)+1))\n for i,x in enumerate(a,1):\n  cur=[i]\n  for j,y in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n  prev=cur\n return prev[-1]\ndef metrics(rs):\n result={}\n for l in [\'overall\',\'de\',\'la\']:\n  sub=[r for r in rs if l==\'overall\' or r[\'language\']==l];chars=sum(len(r[\'reference\']) for r in sub);nw=sum(len(r[\'reference\'].split()) for r in sub);ce=sum(distance(r[\'reference\'],r[\'prediction\']) for r in sub);we=sum(distance(r[\'reference\'].split(),r[\'prediction\'].split()) for r in sub)\n  result[l]=dict(n=len(sub),cer=ce/chars,wer=we/nw,character_errors=ce,reference_chars=chars,word_errors=we,reference_words=nw)\n return result\nbaseline=metrics(rows);assert baseline==json.loads((SOURCE/\'de20/complete.json\').read_text())[\'validation\']\nrecords=[];predictions={}\nfor a,b in weights:\n rs=[];changed=0\n for r in rows:\n  c=max(r[\'candidates\'],key=lambda c:c[\'model_score\']+a*c[\'char_score\']+b*c[\'word_score\']);changed+=c[\'text\']!=r[\'prediction\'];rs.append(dict(line_id=r[\'line_id\'],language=r[\'language\'],reference=r[\'reference\'],prediction=c[\'text\']))\n score=metrics(rs);v=score[\'overall\'];base=baseline[\'overall\'];eligible=v[\'cer\']<=base[\'cer\'] and v[\'wer\']<=base[\'wer\'];key=f\'char{a}_word{b}\'\n record=dict(key=key,char_weight=a,word_weight=b,changed_lines=changed,validation=score,eligible=eligible);records.append(record);predictions[key]=rs;print(\'RESCORE RESULT\',json.dumps(record),flush=True)\nchoices=[r for r in records if r[\'eligible\']];best=min(choices,key=lambda r:max(r[\'validation\'][\'overall\'][\'cer\']/.15,r[\'validation\'][\'overall\'][\'wer\']/.37))\n(OUT/\'summary.json\').write_text(json.dumps(records,indent=2));(OUT/\'selection.json\').write_text(json.dumps(best,indent=2));(OUT/\'selected_validation.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in predictions[best[\'key\']]))\n# Oracle uses validation refs only as a diagnostic ceiling, never a deployed candidate selector.\noracle_words=sum(min(distance(r[\'reference\'].split(),c[\'text\'].split()) for c in r[\'candidates\']) for r in rows);oracle=dict(word_errors=oracle_words,reference_words=2049,wer=oracle_words/2049,diagnostic_only=True)\n(OUT/\'oracle.json\').write_text(json.dumps(oracle));print(\'CANDIDATE ORACLE\',json.dumps(oracle),flush=True);print(\'RESCORING ALL COMPLETE; SELECTED\',json.dumps(best),\'OUTPUT\',OUT,flush=True)\n', 'evaluate_frozen_test.py': '"""Train-text-only character/word LM reranking of beam8 validation candidates."""\nfrom pathlib import Path\nimport json,time,hashlib,zipfile,collections,math,subprocess,sys,shutil,os\nROOT=Path(\'/content/drive/MyDrive/BullingerAblation\');WORK=Path(\'/content/wer-rescoring-test\');WORK.mkdir(exist_ok=True)\nBASE=ROOT/\'runs_scale20k_v1/20261012T195316Z/n20000_natural_s42\'\nSOURCE=ROOT/\'adapt20k_replication_v1/seed44_20261013T172351Z\'\ncfg=json.loads((BASE/\'config.json\').read_text());train=cfg[\'manifest\'][\'train\'];ev=json.loads((BASE/\'evaluation.json\').read_text())\nassert len(train)==20000 and len(ev[\'test\'])==500\nassert hashlib.sha256(json.dumps(ev,sort_keys=True).encode()).hexdigest()==cfg[\'evaluation_sha256\']\nletter=lambda r:Path(r[\'image\']).stem.split(\'_\')[0]\nassert not ({letter(r) for r in train}&{letter(r) for r in ev[\'test\']})\n# Restore only training TXT and frozen test images/TXT for this authorized final test.\nneeded={str(Path(\'dataset\')/r[\'language\']/Path(r[\'text\']).name) for r in train}\nneeded.update(str(Path(\'dataset\')/r[\'language\']/Path(r[k]).name) for r in ev[\'test\'] for k in [\'image\',\'text\'])\nfor name in [\'dataset-baseline.zip\',\'dataset-additional.zip\']:\n with zipfile.ZipFile(ROOT/\'input\'/name) as z:\n  for info in z.infolist():\n   if info.filename in needed:\n    assert (WORK/info.filename).resolve().is_relative_to(WORK.resolve())\n    z.extract(info,WORK);needed.remove(info.filename)\nassert not needed,\'Missing data\'\nOUT=ROOT/\'wer_rescoring_test_v1\'/time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime());OUT.mkdir(parents=True)\npath=lambda r,k:WORK/\'dataset\'/r[\'language\']/Path(r[k]).name\nwords={l:collections.Counter() for l in [\'de\',\'la\']};grams={l:collections.Counter() for l in [\'de\',\'la\']};contexts={l:collections.Counter() for l in [\'de\',\'la\']};alphabet={l:set() for l in [\'de\',\'la\']};digest=hashlib.sha256()\nfor r in train:\n text=path(r,\'text\').read_text(encoding=\'utf-8\').strip();l=r[\'language\'];digest.update(json.dumps([r[\'image\'],l,text],ensure_ascii=False).encode());words[l].update(text.split());alphabet[l].update(text+\'$\')\n s=\'^^\'+text+\'$\'\n for i in range(2,len(s)):grams[l][s[i-2:i+1]]+=1;contexts[l][s[i-2:i]]+=1\n# LM is fitted once, solely on the frozen 20k training references. No modern dictionaries.\ndef lm(text,l):\n s=\'^^\'+text+\'$\';v=len(alphabet[l])+1\n c=sum(math.log((grams[l][s[i-2:i+1]]+.1)/(contexts[l][s[i-2:i]]+.1*v)) for i in range(2,len(s)))/max(1,len(s)-2)\n w=text.split();n=sum(words[l].values());vocab=len(words[l])+1\n u=sum(math.log((words[l][token]+.1)/(n+.1*vocab)) for token in w)/max(1,len(w))\n return c,u\nselected=json.loads((ROOT/\'wer_rescoring_v1/20261013T185538Z/selection.json\').read_text())\nassert selected[\'char_weight\']==selected[\'word_weight\']==.03\nparent_protocol=json.loads((ROOT/\'wer_rescoring_v1/20261013T185538Z/protocol.json\').read_text())\nassert digest.hexdigest()==parent_protocol[\'training_text_sha256\']\nprotocol=dict(parent_protocol,weights=[[.03,.03]],selection=\'frozen validation choice; no test-based tuning\',test_evaluated=True,test_lines=500)\n(OUT/\'protocol.json\').write_text(json.dumps(protocol,indent=2));print(\'FROZEN TEST OUTPUT\',OUT,flush=True)\nsubprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'transformers==4.51.0\',\'Pillow==12.2.0\',\'numpy==2.2.6\'])\nos.environ[\'CUBLAS_WORKSPACE_CONFIG\']=\':4096:8\'\nimport torch\nfrom PIL import Image\nfrom transformers import TrOCRProcessor,VisionEncoderDecoderModel\ntorch.manual_seed(44);torch.use_deterministic_algorithms(True);assert torch.cuda.is_available();print(\'GPU\',torch.cuda.get_device_name(0),flush=True)\nlocal=WORK/\'model\';shutil.copytree(SOURCE/\'de20\',local,ignore=shutil.ignore_patterns(\'training_state.pt\'),dirs_exist_ok=False)\nprocessor=TrOCRProcessor.from_pretrained(local,use_fast=False);model=VisionEncoderDecoderModel.from_pretrained(local).to(\'cuda\').eval();tok=processor.tokenizer\nfor c in [model.config,model.generation_config]:c.decoder_start_token_id=tok.cls_token_id;c.pad_token_id=tok.pad_token_id;c.eos_token_id=tok.sep_token_id\ndef distance(a,b):\n prev=list(range(len(b)+1))\n for i,x in enumerate(a,1):\n  cur=[i]\n  for j,y in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n  prev=cur\n return prev[-1]\ndef metrics(rs):\n result={}\n for l in [\'overall\',\'de\',\'la\']:\n  sub=[r for r in rs if l==\'overall\' or r[\'language\']==l];chars=sum(len(r[\'reference\']) for r in sub);nw=sum(len(r[\'reference\'].split()) for r in sub);ce=sum(distance(r[\'reference\'],r[\'prediction\']) for r in sub);we=sum(distance(r[\'reference\'].split(),r[\'prediction\'].split()) for r in sub)\n  result[l]=dict(n=len(sub),cer=ce/chars,wer=we/nw,character_errors=ce,reference_chars=chars,word_errors=we,reference_words=nw)\n return result\n\ncontrol=[];rescored=[]\nwith (OUT/\'predictions.jsonl\').open(\'w\',encoding=\'utf-8\') as f,torch.inference_mode():\n for i,r in enumerate(ev[\'test\'],1):\n  with Image.open(path(r,\'image\')) as im:x=processor(images=im.convert(\'RGB\'),return_tensors=\'pt\').pixel_values.to(\'cuda\')\n  g=model.generate(x,num_beams=8,num_return_sequences=8,max_new_tokens=128,do_sample=False,use_cache=False,return_dict_in_generate=True,output_scores=True)\n  texts=[t.strip() for t in processor.batch_decode(g.sequences,skip_special_tokens=True)]\n  candidates=[]\n  for t,score in zip(texts,g.sequences_scores.cpu().tolist()):\n   c,w=lm(t,r[\'language\']);candidates.append(dict(text=t,model_score=score,char_score=c,word_score=w))\n  winner=max(candidates,key=lambda c:c[\'model_score\']+.03*c[\'char_score\']+.03*c[\'word_score\'])\n  # Candidate choice is complete before reading this test reference.\n  reference=path(r,\'text\').read_text(encoding=\'utf-8\').strip()\n  row=dict(line_id=Path(r[\'image\']).stem,language=r[\'language\'],reference=reference,prediction=texts[0])\n  control.append(row);rescored.append(dict(row,prediction=winner[\'text\']))\n  f.write(json.dumps(dict(row,rescored_prediction=winner[\'text\'],candidates=candidates),ensure_ascii=False)+\'\\n\');f.flush()\n  if i%25==0:print(\'FROZEN TEST\',i,\'/500\',flush=True)\nresult=dict(control=metrics(control),rescored=metrics(rescored),changed_lines=sum(a[\'prediction\']!=b[\'prediction\'] for a,b in zip(control,rescored)),char_weight=.03,word_weight=.03,test_based_tuning=False)\n(OUT/\'results.json\').write_text(json.dumps(result,indent=2));print(\'FROZEN TEST ALL COMPLETE\',json.dumps(result),\'OUTPUT\',OUT,flush=True)\n', 'context_validation.py': '"""CPU validation-only context LM ablation on frozen saved candidates."""\nfrom pathlib import Path\nimport json,zipfile,collections,math,time,hashlib\nROOT=Path(\'/content/drive/MyDrive/BullingerAblation\');BASE=ROOT/\'runs_scale20k_v1/20261012T195316Z/n20000_natural_s42\';PARENT=ROOT/\'wer_rescoring_v1/20261013T185538Z\'\ntrain=json.loads((BASE/\'config.json\').read_text())[\'manifest\'][\'train\'];rows=[json.loads(s) for s in (PARENT/\'candidates.jsonl\').read_text().splitlines()]\nassert len(train)==20000 and len(rows)==246\nassert not ({Path(r[\'image\']).stem.split(\'_\')[0] for r in train}&{r[\'line_id\'].split(\'_\')[0] for r in rows})\nneeded={str(Path(\'dataset\')/r[\'language\']/Path(r[\'text\']).name) for r in train};texts={}\nprint(\'READING TRAINING TEXT ONLY\',len(needed),flush=True)\nfor archive in [\'dataset-baseline.zip\',\'dataset-additional.zip\']:\n with zipfile.ZipFile(ROOT/\'input\'/archive) as z:\n  for name in z.namelist():\n   if name in needed:texts[name]=z.read(name).decode(\'utf-8\').strip()\nassert len(texts)==len(needed)\ng={l:collections.Counter() for l in [\'de\',\'la\']};ctx={l:collections.Counter() for l in g};alphabet={l:set() for l in g};big={l:collections.Counter() for l in g};wc={l:collections.Counter() for l in g};digest=hashlib.sha256()\nfor r in train:\n t=texts[str(Path(\'dataset\')/r[\'language\']/Path(r[\'text\']).name)];l=r[\'language\'];digest.update(json.dumps([r[\'image\'],l,t],ensure_ascii=False).encode());alphabet[l].update(t+\'$\');s=\'^^^^\'+t+\'$\'\n for i in range(4,len(s)):g[l][s[i-4:i+1]]+=1;ctx[l][s[i-4:i]]+=1\n tokens=[\'<BOS>\']+t.split()+[\'<EOS>\']\n for a,b in zip(tokens,tokens[1:]):big[l][(a,b)]+=1;wc[l][a]+=1\nexpected=json.loads((PARENT/\'protocol.json\').read_text())[\'training_text_sha256\'];assert digest.hexdigest()==expected\nvocab={l:len({b for a,b in big[l]})+1 for l in g}\ndef score(t,l):\n s=\'^^^^\'+t+\'$\';c=sum(math.log((g[l][s[i-4:i+1]]+.1)/(ctx[l][s[i-4:i]]+.1*(len(alphabet[l])+1))) for i in range(4,len(s)))/max(1,len(s)-4)\n ts=[\'<BOS>\']+t.split()+[\'<EOS>\'];w=sum(math.log((big[l][(a,b)]+.1)/(wc[l][a]+.1*vocab[l])) for a,b in zip(ts,ts[1:]))/max(1,len(ts)-1)\n return c,w\nfor r in rows:\n for candidate in r[\'candidates\']:candidate[\'char5\'],candidate[\'word2\']=score(candidate[\'text\'],r[\'language\'])\ndef distance(a,b):\n prev=list(range(len(b)+1))\n for i,x in enumerate(a,1):\n  cur=[i]\n  for j,y in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n  prev=cur\n return prev[-1]\ndef metrics(rs):\n result={}\n for l in [\'overall\',\'de\',\'la\']:\n  sub=[r for r in rs if l==\'overall\' or r[\'language\']==l];ch=sum(len(r[\'reference\']) for r in sub);nw=sum(len(r[\'reference\'].split()) for r in sub);ce=sum(distance(r[\'reference\'],r[\'prediction\']) for r in sub);we=sum(distance(r[\'reference\'].split(),r[\'prediction\'].split()) for r in sub)\n  result[l]=dict(n=len(sub),cer=ce/ch,wer=we/nw,character_errors=ce,reference_chars=ch,word_errors=we,reference_words=nw)\n return result\nout=ROOT/\'context_rescoring_val_v1\'/time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime());out.mkdir(parents=True)\nsettings=[(\'current\',.03,.03),(\'beam8\',0.,0.)]+[(family,w,0.) for family in [\'char5\',\'word2\'] for w in [.01,.03,.1]]+[(\'context_both\',.03,.03)]\n(out/\'protocol.json\').write_text(json.dumps(dict(parent=str(PARENT),training_sha256=digest.hexdigest(),settings=settings,selection=\'both overall CER/WER no worse than current; minimum target worst ratio, retain current on ties\',scope=\'246 validation only; original references; no test read\',known_language_required=True),indent=2))\nrecords=[];predictions={}\nfor family,a,b in settings:\n rs=[]\n for r in rows:\n  def total(c):\n   if family==\'current\':return c[\'model_score\']+a*c[\'char_score\']+b*c[\'word_score\']\n   if family==\'char5\':return c[\'model_score\']+a*c[\'char5\']\n   if family==\'word2\':return c[\'model_score\']+a*c[\'word2\']\n   if family==\'context_both\':return c[\'model_score\']+a*c[\'char5\']+b*c[\'word2\']\n   return c[\'model_score\']\n  winner=max(r[\'candidates\'],key=total);rs.append(dict(line_id=r[\'line_id\'],language=r[\'language\'],reference=r[\'reference\'],prediction=winner[\'text\']))\n key=f\'{family}_{a}_{b}\';record=dict(key=key,validation=metrics(rs),changed_lines=sum(r[\'prediction\']!=n[\'prediction\'] for r,n in zip(rows,rs)));records.append(record);predictions[key]=rs\n if family==\'current\':assert record[\'validation\']==json.loads((PARENT/\'selection.json\').read_text())[\'validation\']\n print(\'CONTEXT RESULT\',json.dumps(record),flush=True)\nbase=records[0][\'validation\'][\'overall\'];eligible=[r for r in records if r[\'validation\'][\'overall\'][\'cer\']<=base[\'cer\'] and r[\'validation\'][\'overall\'][\'wer\']<=base[\'wer\']];best=min(eligible,key=lambda r:max(r[\'validation\'][\'overall\'][\'cer\']/.15,r[\'validation\'][\'overall\'][\'wer\']/.37))\n(out/\'summary.json\').write_text(json.dumps(records,indent=2));(out/\'selection.json\').write_text(json.dumps(best,indent=2));(out/\'selected_validation.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in predictions[best[\'key\']]))\nprint(\'CONTEXT ALL COMPLETE\',json.dumps(best),\'OUTPUT\',out,flush=True)\n'}
BASE=Path("/content/bullinger_delivery")
BASE.mkdir(exist_ok=True)
for name,source in SOURCES.items():(BASE/name).write_text(source)


In [ ]:
if TASK is None:
 print('No task selected. No training or test has started.')
else:
 from google.colab import drive, runtime
 import subprocess, sys
 try:
  drive.mount('/content/drive')
  tasks={'train10k':'train_10k_sampling.py','train20k':'train_20k.py','adapt':'adapt_validation.py','rerank':'rerank_validation.py','context':'context_validation.py','test':'evaluate_frozen_test.py'}
  if TASK not in tasks:raise ValueError('Unknown task')
  command=[sys.executable,'-u',str(BASE/tasks[TASK])]
  if TASK=='adapt':command+=['--seed',str(SEED)]
  subprocess.run(command,check=True)
 finally:
  print('DELIVERY EXIT: RELEASING RUNTIME',flush=True)
  runtime.unassign()
